# Reach direction and velocity tuning in monkey motor cortex

This notebook demonstrates the two classical tuning properties of primate motor cortex during
reaching, using real data streamed from the DANDI Archive:

1. **Direction tuning.** A single neuron's firing rate during a reach is an approximately
   cosine function of the direction of that reach, with a preferred direction that differs
   from neuron to neuron.
2. **Velocity tuning.** The relationship is not just to *which way* the hand goes but to the
   full velocity vector: at a fixed direction, rate scales with hand speed, and the neural
   signal leads the hand by roughly 100 ms.

We then close the loop by decoding: reach direction is recovered from the population with a
median error of about 17 degrees, and instantaneous hand velocity is recovered with a
held-out $R^2$ near 0.7 during movement.

## Dataset

[DANDI:000128](https://dandiarchive.org/dandiset/000128), *MC_Maze*, contributed as part of the
Neural Latents Benchmark (Churchland and Kaufman; Pei et al. 2021). We use
`sub-Jenkins/sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb`, a single session in which
monkey Jenkins performed a delayed reaching task, sometimes straight to a target and sometimes
around virtual barriers ("maze" trials). The file contains 182 sorted units from motor cortex
and hand position and velocity sampled at 1 kHz, over 2295 successful trials.

The 690 MB NWB file is never downloaded in full: it is read over HTTP with `remfile` plus a
local disk cache, and the arrays the analysis actually touches (spike times, hand kinematics,
the trial table) are persisted to a small `.npz` on first run.

**Caveat on brain area.** The units table in this file maps every unit onto electrode rows
0 to 95, all of which are labelled PMd, even though the electrode table describes 96 M1 and 96 PMd
contacts. The mapping therefore cannot be used to separate M1 from PMd, so all units are
treated together as motor cortex.

## Setup

The analysis is split into modules that live next to this notebook:

| module | contents |
|---|---|
| `s01_load.py` | streaming access to the NWB file and the local array cache |
| `s02_behavior.py` | pynapple objects and behavioral quality control |
| `s03_direction_tuning.py` | per-trial cosine tuning and permutation tests |
| `s04_velocity_tuning.py` | continuous velocity tuning, speed sensitivity, lag analysis |
| `s05_glm.py` | NeMoS Poisson GLM model comparison |
| `s06_decoding.py` | population vector and ridge decoding |

In [1]:
import os

import matplotlib
matplotlib.use("Agg")          # this notebook writes figures to disk, it does not display them
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap

import s01_load
import s02_behavior
import s03_direction_tuning as s03
import s04_velocity_tuning as s04
import s05_glm as s05
import s06_decoding as s06

os.makedirs("figures", exist_ok=True)
np.set_printoptions(precision=3, suppress=True)

## 1. Load the session

`load_cache()` streams the NWB file the first time it is called (a few minutes) and reads the
cached arrays thereafter.

In [2]:
raw = s01_load.load_cache()
print("subject:", raw["subject"])
print(raw["session_description"])
print("units:", len(raw["spike_times"]))
print("trials:", len(raw["trial_start"]),
      " barrier-free:", int((raw["trial_num_barriers"] == 0).sum()))
print("behavior samples:", raw["t"].shape[0], "at", 1 / np.median(np.diff(raw["t"])), "Hz")

subject: Jenkins
Data from monkey Jenkins performing center-out delayed reaching task. This file contains trials from the full session on 2009-09-25 that can be used for training models for the Neural Latents Benchmark.
units: 182
trials: 2295  barrier-free: 789
behavior samples: 6809920 at 999.99999990996 Hz


Spikes and kinematics only exist inside trials, so the pynapple `time_support` for every object
is the set of per-trial observation intervals. Reach direction for each trial is defined as the
direction of the mean hand velocity over the first 200 ms of movement, which is well defined for
curved maze reaches as well as straight ones.

In [3]:
S = s02_behavior.build(raw)
print(S["spikes"])
print("\ntotal observed time: %.0f s" % float(S["obs"].tot_length()))
print("mean firing rate: %.2f Hz" % float(np.mean(S["spikes"].rates)))
print("median peak reach speed: %.0f cm/s" % S["trials"]["peak_speed"].median())

  Index      rate
-------  --------
      0   0.85331
      1   0.95772
      2   0.62791
      3   0.22687
      4   1.78284
      5   0.58004
      6   1.77491
      7   1.84907
      8   2.26581
      9   0.16124
     10   0.27342
     11   1.98431
     12   1.2564
     13  10.0577
     14   1.18063
     15   1.52263
     16   1.22615
     17   6.94575
     18   0.54421
     19   0.7119
     20   0.21909
     21   0.66124
     22   1.75626
     23   3.25026
     24   0.57651
     25   0.46564
     26   8.97529
     27   3.10841
     28   0.55243
     29   0.15771
     30   1.08401
     31   0.72292
     32   0.74597
     33   5.43222
     34  10.8738
     35   0.89179
     36   1.8799
     37   1.23526
     38   1.15611
     39   4.07171
     40   2.8761
     41   4.40989
     42   0.89472
     43   1.23907
     44   1.44231
     45   0.61675
     46   3.71752
     47   1.16066
     48   1.18724
     49   1.06976
     50   0.279
     51   0.5583
     52   1.39517
     53   2.88946
 

### Behavioral quality control

Before analysing anything neural, check that the kinematics look like reaches: straight paths on
barrier-free trials, curved paths on maze trials, bell-shaped speed profiles peaking about 130 ms
after the annotated movement onset, and reach directions covering the circle.

In [4]:
s02_behavior.figure_behavior(S)

wrote figures/fig01_behavior_overview.png


![behavior](figures/fig01_behavior_overview.png)

## 2. Direction tuning, trial by trial

For each unit we take the firing rate in a window from 100 ms before to 250 ms after movement
onset (motor cortex leads the hand, so the window is shifted early) and fit

$$r(\theta) = b_0 + b_1 \cos(\theta - \theta_{\text{pref}})$$

by ordinary least squares on $[1, \cos\theta, \sin\theta]$. Significance comes from a
500-shuffle permutation test on the trial direction labels. The same fit is repeated on the
delay period (300 to 50 ms before the go cue), when the target is visible but the monkey has not
yet moved.

In [5]:
R = s03.run(S)
fm, fd = R["fit_move"], R["fit_delay"]
sig = fm["p"] < 0.01
depth = fm["modulation"] / (fm["baseline"] + 1e-9)
print(f"direction-tuned during movement: {sig.sum()}/{len(sig)} units (p < 0.01)")
print(f"direction-tuned during the delay: {(fd['p'] < 0.01).sum()}/{len(sig)} units")
print(f"median modulation depth b1/b0: {np.median(depth[sig]):.2f}")
print(f"median single-trial R^2 of the cosine fit: {np.nanmedian(fm['r2']):.3f}")
Rv, pr = s03.rayleigh(fm["pref_dir"][sig])
print(f"preferred directions: Rayleigh R = {Rv:.2f}, p = {pr:.2f} (uniform around the circle)")

delay-period analysis uses 1999/2295 trials


permutations:   0%|          | 0/500 [00:00<?, ?it/s]

permutations:   0%|          | 0/500 [00:00<?, ?it/s]

direction-tuned during movement: 164/182 units (p < 0.01)
direction-tuned during the delay: 135/182 units
median modulation depth b1/b0: 0.47
median single-trial R^2 of the cosine fit: 0.045
preferred directions: Rayleigh R = 0.07, p = 0.41 (uniform around the circle)


In [6]:
s03.figure_examples(S, R)
s03.figure_population(R)

wrote figures/fig02_direction_tuning_examples.png


wrote figures/fig03_direction_tuning_population.png


![examples](figures/fig02_direction_tuning_examples.png)

Single units show the textbook picture: a broad cosine tuning curve, and a direction-conditioned
PSTH in which the burst for the preferred direction begins *before* the hand starts to move.

![population](figures/fig03_direction_tuning_population.png)

Across the population, preferred directions tile the circle without a strong bias, modulation
depth $b_1/b_0$ is broadly distributed, and the observed modulation is far outside the shuffle
null. The panel comparing delay-period and movement-period preferred directions is worth
noticing: both epochs are strongly direction-tuned, but a unit's preferred direction during
preparation says almost nothing about its preferred direction during the movement itself
(circular correlation near zero). This matches the reported near-orthogonality of preparatory
and perimovement activity in this cortex.

## 3. Velocity tuning, continuously

Trial-averaged direction tuning cannot separate a neuron that codes direction from one that
codes the whole velocity vector. To do that we bin spikes at 20 ms, smooth with a 50 ms Gaussian
to get a rate estimate, bin-average hand velocity onto the same bins, and treat the pair as
continuous signals.

First, the lag. We regress rate on $[1, v_x, v_y]$ for velocity shifted by lags from -500 to
+500 ms and find the lag at which the fit is best.

In [7]:
V = s04.run(S)
print(f"population-optimal lag: {V['pop_lag']*1000:.0f} ms (positive = spikes lead the hand)")
good = V["active"] & (np.nanmax(V["r2_lag"], axis=0) > 0.02)
print(f"median per-unit optimal lag ({good.sum()} well-fit units): "
      f"{np.median(V['best_lag'][good])*1000:.0f} ms")
print(f"adding speed to direction improves held-out R^2 for "
      f"{100*np.mean(V['r2_full'] > V['r2_dironly']):.0f}% of units")

moving epochs: 2388 intervals, 942 s (14% of recorded time)


units with >=500 spikes: 182/182


lag scan:   0%|          | 0/51 [00:00<?, ?it/s]

lag scan:  39%|███▉      | 20/51 [00:15<00:24,  1.27it/s]

lag scan:  39%|███▉      | 20/51 [00:26<00:24,  1.27it/s]

lag scan:  71%|███████   | 36/51 [00:31<00:13,  1.14it/s]

population-optimal lag: 100 ms (positive = neural leads hand)
49304 of 329212 lag-aligned bins have speed > 5.0 cm/s


velocity bins kept: 196/196


population-optimal lag: 100 ms (positive = spikes lead the hand)
median per-unit optimal lag (64 well-fit units): 60 ms
adding speed to direction improves held-out R^2 for 93% of units


In [8]:
s04.figure_2d(V)
s04.figure_speed_and_lag(V)

wrote figures/fig04_velocity_tuning_2d.png


wrote figures/fig05_speed_and_lag.png


![2d](figures/fig04_velocity_tuning_2d.png)

The 2-D tuning surfaces over $(v_x, v_y)$, computed with `nap.compute_tuning_curves` after
shifting velocity by the optimal lag, are graded along the preferred direction rather than
switching on and off with it. Averaged across units after rotating each to its own preferred
direction, the surface has a directional ridge whose height grows with speed.

![speed](figures/fig05_speed_and_lag.png)

The clearest single result is the top-left panel: when the hand moves toward a unit's preferred
direction its rate rises with speed, and when the hand moves the other way its rate falls with
speed. Direction alone cannot produce that. The nested, cross-validated regression confirms it,
and the lag analysis puts the population peak at +100 ms with a broad per-unit distribution.

One thing to rule out is double dipping: each unit's preferred direction was estimated from the
same bins used to measure the speed split, so a unit whose preferred direction was set by noise
could show a spurious split. The control below estimates preferred directions on the first half
of the moving bins and measures the speed slopes on the second half.

In [9]:
ctl = s04.speed_tuning_control(S)
print(f"slope toward the preferred direction: median {np.nanmedian(ctl['slope_pref']):+.4f} "
      f"Hz per cm/s, positive for {100*ctl['frac_pos_pref']:.0f}% of units")
print(f"slope away from the preferred direction: median {np.nanmedian(ctl['slope_anti']):+.4f} "
      f"Hz per cm/s, negative for {100*ctl['frac_neg_anti']:.0f}% of units")
print(f"Wilcoxon signed-rank, toward vs away: p = {ctl['wilcoxon_p']:.1e}")

slope toward the preferred direction: median +0.0108 Hz per cm/s, positive for 70% of units
slope away from the preferred direction: median -0.0031 Hz per cm/s, negative for 75% of units
Wilcoxon signed-rank, toward vs away: p = 2.6e-18


## 4. A Poisson GLM (NeMoS)

The regressions above are linear-Gaussian and impose a cosine shape. A Poisson GLM with flexible
bases removes both assumptions. Five feature sets are compared on held-out contiguous blocks,
scored by McFadden pseudo-$R^2$:

* `constant`: intercept only
* `speed`: B-spline over hand speed
* `direction`: cyclic B-spline over the direction of the velocity vector
* `dir + speed`: the two additively
* `dir x speed`: their outer product, an unconstrained velocity tuning surface

This cell refits the models if `cache/glm_results.npz` is absent, which takes roughly five
minutes.

In [10]:
G = s05.run_cached(S)
for name, sc in G["scores"].items():
    print(f"  {name:12s} median cross-validated pseudo-R^2 = {np.median(sc):.4f}")
print(f"\ndir x speed beats direction alone for "
      f"{100*np.mean(G['scores']['dir x speed'] > G['scores']['direction']):.0f}% of units")

loaded cached GLM results from cache/glm_results.npz
  constant     median cross-validated pseudo-R^2 = 0.0000
  speed        median cross-validated pseudo-R^2 = 0.0025
  direction    median cross-validated pseudo-R^2 = 0.0114
  dir + speed  median cross-validated pseudo-R^2 = 0.0155
  dir x speed  median cross-validated pseudo-R^2 = 0.0198

dir x speed beats direction alone for 95% of units


In [11]:
s05.figure_glm(G)

wrote figures/fig06_glm_encoding.png


![glm](figures/fig06_glm_encoding.png)

The ordering is unambiguous: speed alone is a weak predictor, direction alone is roughly four
times better, and the two together are better than either. Allowing direction and speed to
interact adds a further increment for 95% of units. The example polar plots show what the
interaction buys: the tuning curve grows with speed, and for some units it also rotates.

## 5. Decoding

Two decoders, one classical and one statistical.

The **population vector** needs nothing but the preferred directions already measured: sum each
tuned unit's preferred direction unit vector weighted by its z-scored rate on that trial.

The **ridge decoder** regresses instantaneous hand velocity on a 0 to 80 ms window of smoothed
population activity, cross-validated over five contiguous blocks.

In [12]:
D = s06.run(S, R)   # reuse the direction fits from section 2

population vector: 164 tuned units, median |error| = 16.8 deg


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-4-8/reach-02/s06_decoding.py:32: RuntimeWarning: divide by zero encountered in matmul
  vec = z @ np.column_stack([np.cos(pd), np.sin(pd)])
/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-4-8/reach-02/s06_decoding.py:32: RuntimeWarning: overflow encountered in matmul
  vec = z @ np.column_stack([np.cos(pd), np.sin(pd)])
/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-4-8/reach-02/s06_decoding.py:32: RuntimeWarning: invalid value encountered in matmul
  vec = z @ np.column_stack([np.cos(pd), np.sin(pd)])


ridge folds:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: overflow encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_.T + self.intercept_


...

/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: overflow encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_.T + self.intercept_


/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: overflow encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/bdichter/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_.T + self.intercept_



population size sweep: 100%|██████████| 8/8 [03:17<00:00, 35.14s/it]

population size sweep: 100%|██████████| 8/8 [03:17<00:00, 24.75s/it]

In [13]:
s06.figure_decoding(S, D)

wrote figures/fig07_decoding.png


![decoding](figures/fig07_decoding.png)

The population vector recovers the reach direction of single trials with a median absolute error
of about 17 degrees against a chance level of 90. The ridge decoder tracks both components of
hand velocity through the reach, and the decoded speed correlates with the true speed at
$r \approx 0.73$, which is the decoding-side statement of the same fact the tuning curves showed:
the population carries speed, not only direction. Decoding accuracy climbs steadily with the
number of units included and has not saturated at 182.

## Summary

* 164 of 182 motor cortical units in this session are significantly cosine-tuned to reach
  direction (permutation test, p < 0.01), with preferred directions spread uniformly around the
  circle and a median modulation depth of about 0.4 of baseline rate.
* The tuning is to velocity, not merely direction. Firing rate increases with hand speed for
  movements toward the preferred direction and decreases for movements away from it, and adding
  speed to a direction-only model improves held-out prediction for 93% of units in a linear
  model and 95% in a Poisson GLM.
* The neural signal leads the hand. The velocity model fits best when spiking is compared with
  hand velocity 100 ms later.
* Preparatory (delay-period) direction tuning is strong but its preferred directions are
  essentially uncorrelated with the movement-period preferred directions of the same units.
* Both quantities can be read back out: 17 degrees median error for single-trial reach direction
  from a population vector, and held-out $R^2 \approx 0.75/0.69$ for $v_x/v_y$ during movement
  from a ridge decoder.